# CS781 Phase 7 — Experiment 3
## Phase-1 Model 2 + NF4 QLoRA Cross-Encoder for Prior Case Retrieval

- Starts from the **original Phase-1 Model 2 decoder-only checkpoint** and its matching LegalBPE tokenizer.
- Sparse-matrix BM25 hard-negative mining uses only the Train candidate pool; Dev/Test pools remain isolated. Retrieval uses exact LegalBPE token IDs.
- Query-conditioned causal cross-encoder scores query–candidate pairs.
- NF4 quantized linear projections + rank-32 LoRA adapters; frozen token embedding/output matrix.
- Combined BCE + pairwise + listwise ranking objective; hard negatives are re-sampled each epoch.
- Four epoch snapshots are evaluated on Dev; the best checkpoint and BM25/neural fusion weight are selected by the **professor's official Micro-F1@7 evaluator**.
- Final ZIP contains `prediction.json` at its root and is structurally checked with the same official evaluator code.

### Runtime estimate (Kaggle T4; planning estimate, not benchmarked)
- Setup, checkpoint/tokenizer loading, data download + EDA + sparse BM25 indexing: **8–18 min**
- LegalBPE tokenization cache + sparse BM25 hard-negative mining: **2–8 min (estimate; dataset dependent)**
- Training: **4 epochs × roughly 25–50 min = 100–200 min**
- Dev scoring/checkpoint selection: **30–80 min** (four snapshots; depends on Dev size)
- Test inference + official structural validation + ZIP: **10–30 min**
- **Total planning range: about 2.5–6 hours on a T4.** Run the QLoRA forward/backward smoke test before starting the full training loop. Actual runtime depends heavily on dataset size and Kaggle GPU availability.

All output artifacts are kept in one experiment folder: `/kaggle/working/phase7_model2_qlora/`.


In [24]:
import os, sys, json, math, time, random, gc, re, subprocess, zipfile, hashlib, heapq, pickle
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
# Fast sparse-matrix BM25, adapted from the earlier Model 3 retrieval experiment.
# Documents and queries are still tokenized ONLY with the exact Phase-1 LegalBPE.
# This avoids the Python loop over every document for every query.
from scipy import sparse

class BM25Okapi:
    def __init__(self, corpus, k1=1.35, b=0.72):
        self.k1, self.b = float(k1), float(b)
        self.n_docs = len(corpus)
        self.doc_len = np.asarray([len(doc) for doc in corpus], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.n_docs else 1.0

        # Build a compact document-term CSR matrix from integer LegalBPE token IDs.
        term_to_col = {}
        rows, cols, values = [], [], []
        df = Counter()
        for row, doc in enumerate(corpus):
            freq = Counter(int(t) for t in doc)
            for term, tf in freq.items():
                col = term_to_col.get(term)
                if col is None:
                    col = len(term_to_col)
                    term_to_col[term] = col
                rows.append(row)
                cols.append(col)
                values.append(float(tf))
            df.update(freq.keys())

        self.term_to_col = term_to_col
        n_terms = len(term_to_col)
        counts = sparse.csr_matrix(
            (np.asarray(values, dtype=np.float32),
             (np.asarray(rows, dtype=np.int32), np.asarray(cols, dtype=np.int32))),
            shape=(self.n_docs, n_terms), dtype=np.float32
        )

        if self.n_docs and n_terms:
            df_arr = np.zeros(n_terms, dtype=np.float32)
            for term, col in term_to_col.items():
                df_arr[col] = df[term]
            # Same positive BM25 IDF form as the earlier sparse BM25 retrieval code.
            idf = np.log1p((self.n_docs - df_arr + 0.5) / (df_arr + 0.5)).astype(np.float32)
            row_ids = np.repeat(np.arange(self.n_docs), np.diff(counts.indptr))
            tf = counts.data
            norm = self.k1 * (
                1.0 - self.b + self.b * self.doc_len[row_ids] / max(self.avgdl, 1.0)
            )
            weighted = idf[counts.indices] * (tf * (self.k1 + 1.0) / (tf + norm))
            self.matrix = sparse.csr_matrix(
                (weighted.astype(np.float32), counts.indices.copy(), counts.indptr.copy()),
                shape=counts.shape
            )
        else:
            self.matrix = sparse.csr_matrix((self.n_docs, n_terms), dtype=np.float32)

    def get_scores(self, query):
        # Count repeated query tokens, matching standard bag-of-token BM25 scoring.
        qfreq = Counter(int(t) for t in query)
        if not qfreq or self.matrix.shape[1] == 0:
            return np.zeros(self.n_docs, dtype=np.float32)
        qcols, qvals = [], []
        for term, count in qfreq.items():
            col = self.term_to_col.get(term)
            if col is not None:
                qcols.append(col)
                qvals.append(float(count))
        if not qcols:
            return np.zeros(self.n_docs, dtype=np.float32)
        qvec = sparse.csr_matrix(
            (np.asarray(qvals, dtype=np.float32),
             (np.asarray(qcols, dtype=np.int32), np.zeros(len(qcols), dtype=np.int32))),
            shape=(self.matrix.shape[1], 1), dtype=np.float32
        )
        return np.asarray((self.matrix @ qvec).toarray()).ravel().astype(np.float32)

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Please select a Kaggle GPU accelerator (T4 or better)."
print("GPU:", torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

REPO_ID = "Exploration-Lab/CS-781-Capstone"
ROOT = Path("/kaggle/working/phase7_model2_qlora")
ROOT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "cache"; CACHE.mkdir(exist_ok=True)
CKPT_DIR = ROOT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
PRED_DIR = ROOT / "predictions"; PRED_DIR.mkdir(exist_ok=True)

# Set these only if automatic Kaggle input discovery finds multiple matches.
CHECKPOINT_PATH = None  # e.g. "/kaggle/input/.../dec_full_best (8).pt"
TOKENIZER_PATH = None   # e.g. "/kaggle/input/.../tokenizer.json"

# Retrieval/training controls — deliberately conservative for a T4.
EPOCHS = 4                  # was 3
TOP_K_BM25_TRAIN = 60       # hard-negative POOL per query (was 40); negatives are re-sampled every epoch
TOP_K_BM25_EVAL = 100
MAX_TRAIN_GROUPS = None     # None = all train queries
NEGATIVES_PER_QUERY = 7     # was 6
POS_PER_QUERY = 3
HARD_FRAC = 0.5             # half of the negatives from the hardest 15 BM25 hits, half from the rest of the pool
PAIR_MAX_LEN = 512          # was 448
QUERY_MAX_TOKENS = 160
CAND_MAX_TOKENS = PAIR_MAX_LEN - QUERY_MAX_TOKENS - 3
BATCH_GROUPS = 2  # 2 query groups per forward (lower to 1 if CUDA OOM)
EVAL_BATCH_SIZE = 16
GRAD_ACCUM = 4    # effective batch = 8 query groups (was 4)
LR_LORA = 1.5e-4  # was 8e-5
LR_HEAD = 8e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
RANK = 32         # was 16
ALPHA = 64
DROPOUT = 0.05
RETRIEVAL_TOP_K = 100
SEED = 2026

def stamp():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print("Output:", ROOT, "|", stamp())
print("Runtime planning estimate (T4, not benchmarked): 2.5–6 hours total;")
print("  setup+EDA+BM25 10–25m | hard-negative prep 10–25m | training 4x25–50m")
print("  Dev snapshot scoring 30–80m | Test+validation+ZIP 10–30m")

GPU: Tesla T4
Output: /kaggle/working/phase7_model2_qlora | 2026-11-06 13:22:45
Runtime planning estimate (T4, not benchmarked): 2.5–6 hours total;
  setup+EDA+BM25 10–25m | hard-negative prep 10–25m | training 4x25–50m
  Dev snapshot scoring 30–80m | Test+validation+ZIP 10–30m


## 1. Resolve Phase-1 checkpoint and tokenizer

The checkpoint and tokenizer are expected in Kaggle Inputs. This cell refuses ambiguous discovery rather than silently loading the wrong architecture.

In [25]:
def find_one(patterns, explicit, label):
    if explicit:
        p = Path(explicit)
        if not p.is_file(): raise FileNotFoundError(f"{label} not found: {p}")
        return p
    hits = []
    for pat in patterns:
        hits += list(Path("/kaggle/input").rglob(pat))
    hits = sorted({p.resolve() for p in hits if p.is_file()})
    if len(hits) != 1:
        print(label, "candidates:")
        for p in hits[:30]: print(" -", p)
        raise RuntimeError(f"Expected exactly one {label}; set its explicit path at the top.")
    return hits[0]

CHECKPOINT_PATH = find_one(["dec_full_best*.pt"], CHECKPOINT_PATH, "Model-2 best checkpoint")
TOKENIZER_PATH = find_one(["tokenizer (1).json"], TOKENIZER_PATH, "Phase-1 tokenizer.json")
print("Checkpoint:", CHECKPOINT_PATH, f"({CHECKPOINT_PATH.stat().st_size/1e9:.2f} GB)")
print("Tokenizer:", TOKENIZER_PATH)

Checkpoint: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt (0.32 GB)
Tokenizer: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json


In [26]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
print("Tokenizer vocab:", len(tokenizer))
assert len(tokenizer) == 24576, f"Unexpected Model-2 vocabulary: {len(tokenizer)}"
probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(probe)) == probe
print("Tokenizer round-trip: PASS")

Tokenizer class ready.
Tokenizer vocab: 24576
Tokenizer round-trip: PASS


## 2. Load the Phase-1 Model 2 checkpoint exactly

No architecture is re-created from guessed defaults: configuration is read from the checkpoint and strict state-dict loading is required.

In [27]:
# Cell 9 — ~80M Decoder-only causal Transformer with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        # Training / ordinary prefill: use SDPA's efficient causal path.
        # Left-padded batched prefill needs an explicit causal + key-valid mask.
        # Cached one-token decoding already has only past/current keys, so a key-valid mask is enough.
        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :]  # (B,1,1,S), True = real key
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]

                # Left-padding creates early PAD query rows with no valid causal key.
                # Giving each such row access to itself avoids fully-masked SDPA rows
                # on kernels/versions that may otherwise produce NaNs. Real-token
                # attention is unchanged because real positions already include self.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-style scaled residual projections.
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_ids=None,
        key_valid=None,
    ):
        B, T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        # Tied token/output embedding.
        return F.linear(hidden, self.token_emb.weight)




def safe_torch_load(path):
    try: return torch.load(path, map_location="cpu", weights_only=True)
    except Exception: return torch.load(path, map_location="cpu", weights_only=False)

ckpt = safe_torch_load(CHECKPOINT_PATH)
assert isinstance(ckpt, dict) and "model" in ckpt and "config" in ckpt, list(ckpt)[:20]
cfg = dict(ckpt["config"])
D_MODEL = int(cfg.get("d_model", cfg.get("D_MODEL", 640)))
N_LAYERS = int(cfg.get("n_layers", cfg.get("N_LAYERS", 13)))
N_HEADS = int(cfg.get("n_heads", cfg.get("N_HEADS", 10)))
FFN_DIM = int(cfg.get("ffn_dim", cfg.get("FFN_DIM", 2560)))
DROPOUT_BASE = float(cfg.get("dropout", cfg.get("DROPOUT", 0.0)))
MAX_POSITIONS = int(cfg.get("max_positions", cfg.get("MAX_POSITIONS", 2048)))
assert int(cfg.get("vocab_size", len(tokenizer))) == len(tokenizer)
assert PAIR_MAX_LEN <= MAX_POSITIONS

model = DecoderOnlyLM(len(tokenizer), D_MODEL, N_LAYERS, N_HEADS, FFN_DIM,
                      DROPOUT_BASE, MAX_POSITIONS)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
model.to(DEVICE)
print(f"Loaded Phase-1 Model 2: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")
print("Config:", {"d_model":D_MODEL,"layers":N_LAYERS,"heads":N_HEADS,
                  "ffn":FFN_DIM,"max_positions":MAX_POSITIONS})
del ckpt; gc.collect()

Loaded Phase-1 Model 2: 79.74M params
Config: {'d_model': 640, 'layers': 13, 'heads': 10, 'ffn': 2560, 'max_positions': 2048}


8735

## 3. NF4 + LoRA conversion

The custom decoder is not a Hugging Face `PreTrainedModel`; this conversion operates directly on its `nn.Linear` modules. We freeze the Phase-1 base weights, quantize linear projections to NF4, and train LoRA adapters plus a small ranking head.

In [28]:
# Install bitsandbytes for 4-bit QLoRA
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "bitsandbytes"
])

import bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit, Params4bit

print("bitsandbytes version:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

bitsandbytes version: 0.50.2
CUDA available: True
GPU: Tesla T4


In [29]:
# bitsandbytes is required for true 4-bit NF4. Do not silently fall back to fake quantization.
try:
    import bitsandbytes as bnb
    from bitsandbytes.nn import Linear4bit, Params4bit
except Exception as e:
    raise RuntimeError("Install/enable bitsandbytes in Kaggle before continuing.") from e

class QLoRALinear(nn.Module):
    def __init__(self, base4, rank=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base4
        for p in self.base.parameters(): p.requires_grad = False
        self.rank = rank
        self.scale = alpha / rank
        self.lora_A = nn.Parameter(torch.empty(rank, base4.in_features, device=DEVICE, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(base4.out_features, rank, device=DEVICE, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        z = F.linear(self.dropout(x).float(), self.lora_A)
        lora_out = F.linear(z, self.lora_B) * self.scale
        return base_out + lora_out.to(base_out.dtype)

def quantize_linear(old):
    new = Linear4bit(old.in_features, old.out_features, bias=(old.bias is not None),
                     compute_dtype=torch.float16, compress_statistics=True, quant_type="nf4")
    # Assign original full-precision weights as Params4bit; .to(cuda) triggers quantization.
    new.weight = Params4bit(old.weight.detach().cpu(), requires_grad=False,
                            compress_statistics=True, quant_type="nf4")
    if old.bias is not None:
        new.bias = nn.Parameter(old.bias.detach().to(torch.float16), requires_grad=False)
    new = new.to(DEVICE)
    return QLoRALinear(new, RANK, ALPHA, DROPOUT)

def replace_target_linears(module, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        full = f"{prefix}.{name}" if prefix else name
        # All attention and MLP projections are adapted; embeddings and LayerNorm stay frozen.
        if isinstance(child, nn.Linear):
            setattr(module, name, quantize_linear(child))
            replaced.append(full)
        else:
            replaced.extend(replace_target_linears(child, full))
    return replaced

for p in model.parameters(): p.requires_grad = False
replaced_names = replace_target_linears(model)
assert replaced_names, "No Linear projections were quantized."
# Frozen tied embedding/output remains FP16 to avoid a large vocabulary matrix quantization issue.
model.token_emb.weight.data = model.token_emb.weight.data.to(torch.float16)
print("NF4 projections:", len(replaced_names))
print("First targets:", replaced_names[:12])
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, "M")

NF4 projections: 52
First targets: ['blocks.0.attn.qkv', 'blocks.0.attn.out_proj', 'blocks.0.ff.0', 'blocks.0.ff.2', 'blocks.1.attn.qkv', 'blocks.1.attn.out_proj', 'blocks.1.ff.0', 'blocks.1.ff.2', 'blocks.2.attn.qkv', 'blocks.2.attn.out_proj', 'blocks.2.ff.0', 'blocks.2.ff.2']
Trainable params: 4.25984 M


## 4. Phase-7 data and BM25 hard-negative mining

Each split uses its own candidate pool. The BM25 index for training is built only over `train_candidates.json`; Dev/Test never contribute training negatives.

In [30]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, hf_hub_download

# Read token from Kaggle Secrets
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), \
    "HF_TOKEN is missing or invalid. Check the Kaggle Secret name."

# Authenticate explicitly
login(token=HF_TOKEN, add_to_git_credential=False)

# Verify which Hugging Face account is authenticated
api = HfApi(token=HF_TOKEN)
print("Authenticated HF account:", api.whoami()["name"])

# Verify access to the dataset repository
repo_info = api.dataset_info("Exploration-Lab/CS-781-Capstone")
print("Dataset access verified:", repo_info.id)

Authenticated HF account: A-230023
Dataset access verified: Exploration-Lab/CS-781-Capstone


In [31]:
def load_hf_json(rel):
    path = hf_hub_download(repo_id=REPO_ID, filename=rel, repo_type="dataset", token=HF_TOKEN)
    with open(path, encoding="utf-8") as f: return json.load(f)

train_q = load_hf_json("Phase_7_PCR/Train/train.json")
train_c = load_hf_json("Phase_7_PCR/Train/train_candidates.json")
dev_q = load_hf_json("Phase_7_PCR/Dev/dev.json")
dev_c = load_hf_json("Phase_7_PCR/Dev/dev_candidates.json")
test_q = load_hf_json("Phase_7_PCR/Test/test.json")
test_c = load_hf_json("Phase_7_PCR/Test/test_candidates.json")

def norm_ids(d): return {str(k):v for k,v in d.items()}
train_q,train_c,dev_q,dev_c,test_q,test_c = map(norm_ids,[train_q,train_c,dev_q,dev_c,test_q,test_c])

def txt(r):
    x=r.get("text","")
    return " ".join(map(str,x)) if isinstance(x,list) else str(x)
# Use the exact Phase-1 Model 2 LegalBPE tokenizer for ALL lexical retrieval.
# BM25 accepts integer token IDs; no regex/whitespace tokenizer is used.
def toks(s):
    return tokenizer.encode(str(s))

for name,q,c in [("Train",train_q,train_c),("Dev",dev_q,dev_c),("Test",test_q,test_c)]:
    print(name, "queries",len(q),"candidates",len(c))
    assert q and c

# EDA: query/candidate lengths and label coverage
train_lens=[len(txt(v)) for v in train_q.values()]
cand_lens=[len(txt(v)) for v in train_c.values()]
print("Train query chars p50/p90:", np.percentile(train_lens,[50,90]).astype(int))
print("Train candidate chars p50/p90:", np.percentile(cand_lens,[50,90]).astype(int))
label_counts=[len(v.get("relevant_candidates",[])) for v in train_q.values()]
print("Train relevant labels/query p50/p90:", np.percentile(label_counts,[50,90]))

Train queries 827 candidates 4320
Dev queries 118 candidates 1023
Test queries 237 candidates 1727
Train query chars p50/p90: [25893 72937]
Train candidate chars p50/p90: [20848 71744]
Train relevant labels/query p50/p90: [ 4. 14.]


In [32]:
# ============================================================
# LEGALBPE BM25 INDEX + CACHED TOKENIZATION + HARD NEGATIVE MINING
# Uses the exact Phase-1 Model 2 LegalBPE tokenizer.
# Sparse CSR BM25 follows the optimized retrieval approach used in the earlier Model 3 experiment.
# ============================================================

t_bm25 = time.time()
train_cids = list(train_c.keys())

TOKEN_CACHE_PATH = CACHE / "train_candidate_legalbpe_tokens.pkl"
train_ctoks = None

# Cache validity checks both candidate IDs/order and tokenizer identity.
tokenizer_fingerprint = hashlib.sha256(
    json.dumps(
        {"vocab_size": len(tokenizer), "merges": tokenizer.merges},
        ensure_ascii=False
    ).encode("utf-8")
).hexdigest()

if TOKEN_CACHE_PATH.exists():
    try:
        with open(TOKEN_CACHE_PATH, "rb") as f:
            token_cache = pickle.load(f)
        if (
            token_cache.get("candidate_ids") == train_cids
            and token_cache.get("tokenizer_fingerprint") == tokenizer_fingerprint
            and len(token_cache.get("tokens", [])) == len(train_cids)
        ):
            train_ctoks = token_cache["tokens"]
            print(f"Loaded {len(train_ctoks):,} LegalBPE-tokenized candidates from cache.")
        else:
            print("Token cache does not match this candidate pool/tokenizer; rebuilding.")
    except Exception as exc:
        print("Could not read token cache; rebuilding:", repr(exc))

if train_ctoks is None:
    print(f"Tokenizing {len(train_cids):,} training candidates with LegalBPE...")
    train_ctoks = []
    for i, cid in enumerate(train_cids, 1):
        train_ctoks.append(toks(txt(train_c[cid])))
        if i % 500 == 0 or i == len(train_cids):
            print(
                f"LegalBPE tokenized {i:,}/{len(train_cids):,} | "
                f"elapsed {(time.time()-t_bm25)/60:.1f} min",
                flush=True
            )

    with open(TOKEN_CACHE_PATH, "wb") as f:
        pickle.dump(
            {
                "candidate_ids": train_cids,
                "tokenizer_fingerprint": tokenizer_fingerprint,
                "tokens": train_ctoks,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    print("LegalBPE token cache saved:", TOKEN_CACHE_PATH)

assert len(train_ctoks) == len(train_cids)
assert all(isinstance(doc, list) for doc in train_ctoks)

print("Building sparse Train BM25 index over LegalBPE token IDs...")
bm25_train = BM25Okapi(train_ctoks)
cid_to_idx = {cid: i for i, cid in enumerate(train_cids)}

def bm25_rank(query, bm25, ids, topk):
    # Query is encoded with the SAME LegalBPE tokenizer as the documents.
    query_tokens = toks(query)
    scores = bm25.get_scores(query_tokens)
    k = min(int(topk), len(scores))
    if k <= 0:
        return []
    # Partial selection followed by deterministic descending score sort.
    ix = np.argpartition(scores, -k)[-k:]
    ix = sorted(ix.tolist(), key=lambda j: scores[j], reverse=True)
    return [(ids[j], float(scores[j])) for j in ix]

print(f"BM25 index ready in {(time.time()-t_bm25)/60:.2f} min.")
print("Mining LegalBPE-BM25 hard negatives with sparse matrix scoring...")

train_groups = []
rng = random.Random(SEED)
total_queries = len(train_q)
t_mine = time.time()

for qi, (qid, qrec) in enumerate(train_q.items(), 1):
    positives = [
        str(x) for x in qrec.get("relevant_candidates", [])
        if str(x) in train_c
    ]
    if not positives:
        continue

    positive_set = set(positives)
    ranked = bm25_rank(
        txt(qrec), bm25_train, train_cids, TOP_K_BM25_TRAIN
    )
    negs = [cid for cid, _ in ranked if cid not in positive_set]

    if len(negs) < NEGATIVES_PER_QUERY:
        neg_set = set(negs)
        extra = [
            cid for cid in train_cids
            if cid not in positive_set and cid not in neg_set
        ]
        rng.shuffle(extra)
        negs.extend(extra)

    # Retain labeled positives and the full BM25 hard-negative pool.
    selected = positives[:POS_PER_QUERY] + negs[:TOP_K_BM25_TRAIN]
    if len(selected) > 1:
        train_groups.append((qid, txt(qrec), selected, positive_set))

    if qi % 100 == 0 or qi == total_queries:
        elapsed = time.time() - t_mine
        rate = qi / max(elapsed, 1e-8)
        eta_min = (total_queries - qi) / max(rate, 1e-8) / 60
        print(
            f"Mining {qi:,}/{total_queries:,} | groups={len(train_groups):,} | "
            f"elapsed={elapsed/60:.1f} min | ETA={eta_min:.1f} min",
            flush=True
        )

if MAX_TRAIN_GROUPS is not None:
    train_groups = train_groups[:MAX_TRAIN_GROUPS]

print("=" * 65)
print("LEGALBPE BM25 + HARD-NEGATIVE MINING COMPLETE")
print("Train candidates:", len(train_cids))
print("Train queries:", total_queries)
print("Training groups:", len(train_groups))
print("LegalBPE vocab:", len(tokenizer))
print(f"Total BM25 preparation time: {(time.time()-t_bm25)/60:.2f} min")
assert train_groups, "No valid training groups were generated."


Loaded 4,320 LegalBPE-tokenized candidates from cache.
Building sparse Train BM25 index over LegalBPE token IDs...
BM25 index ready in 0.20 min.
Mining LegalBPE-BM25 hard negatives with sparse matrix scoring...
Mining 100/827 | groups=99 | elapsed=0.1 min | ETA=0.8 min
Mining 200/827 | groups=197 | elapsed=0.2 min | ETA=0.7 min
Mining 300/827 | groups=295 | elapsed=0.3 min | ETA=0.5 min
Mining 400/827 | groups=394 | elapsed=0.4 min | ETA=0.4 min
Mining 500/827 | groups=494 | elapsed=0.5 min | ETA=0.3 min
Mining 600/827 | groups=594 | elapsed=0.6 min | ETA=0.2 min
Mining 700/827 | groups=691 | elapsed=0.7 min | ETA=0.1 min
Mining 800/827 | groups=790 | elapsed=0.8 min | ETA=0.0 min
Mining 827/827 | groups=817 | elapsed=0.8 min | ETA=0.0 min
LEGALBPE BM25 + HARD-NEGATIVE MINING COMPLETE
Train candidates: 4320
Train queries: 827
Training groups: 817
LegalBPE vocab: 24576
Total BM25 preparation time: 1.01 min


## 5. Pair construction and causal cross-encoder scoring

The model sees the query first and the candidate second. The candidate-side hidden states are pooled for a scalar relevance score. The query remains in the causal context, so the candidate representation is query-conditioned.

In [33]:
_TOK_CACHE={}
def _tok_cached(text):
    key=(len(text),hash(text))
    hit=_TOK_CACHE.get(key)
    if hit is None:
        hit=tokenizer.encode(text)
        if len(_TOK_CACHE)>6000: _TOK_CACHE.clear()
        _TOK_CACHE[key]=hit
    return hit

def head_tail(ids, limit, head_frac=0.7):
    # Keep the start (facts/parties) AND the end (holding/outcome) of long legal documents.
    if len(ids)<=limit: return ids
    h=int(limit*head_frac)
    return ids[:h]+ids[-(limit-h):]

def encode_pair(qtext, ctext):
    qids=head_tail(_tok_cached(qtext),QUERY_MAX_TOKENS)
    cids=head_tail(_tok_cached(ctext),CAND_MAX_TOKENS)
    ids=[tokenizer.bos_token_id]+qids+[tokenizer.eos_token_id]+cids+[tokenizer.eos_token_id]
    ids=ids[-PAIR_MAX_LEN:]
    # Candidate positions begin after query + separator; ensure at least one candidate token.
    cand_start=min(len(ids)-1, 1+len(qids)+1)
    return ids,cand_start

def collate_pairs(items):
    # items: [(query_text, candidate_text, label, group_id)]
    enc=[encode_pair(x[0],x[1]) for x in items]
    L=max(len(z[0]) for z in enc)
    ids=torch.full((len(items),L),tokenizer.pad_token_id,dtype=torch.long)
    mask=torch.zeros((len(items),L),dtype=torch.bool)
    starts=[]
    for i,(seq,st) in enumerate(enc):
        ids[i,:len(seq)]=torch.tensor(seq)
        mask[i,:len(seq)]=True
        starts.append(min(st,len(seq)-1))
    return ids.to(DEVICE),mask.to(DEVICE),torch.tensor(starts,device=DEVICE),torch.tensor([x[2] for x in items],dtype=torch.float32,device=DEVICE),[x[3] for x in items]

class RankHead(nn.Module):
    def __init__(self,d):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d*2,d),nn.GELU(),nn.Dropout(0.1),nn.Linear(d,1))
    def forward(self,q,c):
        return self.net(torch.cat([q,c],dim=-1)).squeeze(-1)

rank_head=RankHead(D_MODEL).to(DEVICE)

In [34]:
def decoder_pair_scores(ids, mask, starts):
    # Left padding is not used; position IDs are standard 0..L-1 for each row.
    pos=torch.arange(ids.size(1),device=DEVICE)[None,:].expand_as(ids)
    with torch.autocast("cuda",dtype=torch.float16):
        h,_=model.forward_hidden(ids,position_ids=pos,key_valid=mask)
    # Query vector: hidden state at separator; candidate vector: mean over candidate tokens.
    qpos=(starts-1).clamp(min=0)
    qvec=h[torch.arange(len(starts),device=DEVICE),qpos].float()
    ix=torch.arange(ids.size(1),device=DEVICE)[None,:]
    cmask=(ix>=starts[:,None]) & mask
    cvec=(h.float()*cmask[:,:,None]).sum(1)/cmask.sum(1,keepdim=True).clamp(min=1)
    return rank_head(qvec,cvec)

# Smoke test: forward + backward before any expensive epoch.
smoke=[]
for qid,qtext,cids,posset in train_groups[:1]:
    for cid in cids[:4]:
        smoke.append((qtext,txt(train_c[cid]),float(cid in posset),qid))
ids,mask,starts,labels,gids=collate_pairs(smoke)
sc=decoder_pair_scores(ids,mask,starts)
loss=F.binary_cross_entropy_with_logits(sc,labels)
loss.backward()
assert torch.isfinite(loss)
print("Smoke test PASS | scores:",sc.detach().float().cpu().tolist(),"loss:",float(loss))
model.zero_grad(set_to_none=True); rank_head.zero_grad(set_to_none=True)

Smoke test PASS | scores: [0.18416090309619904, 0.6370356678962708, 0.022379685193300247, -0.009840765036642551] loss: 0.6056149005889893


/tmp/ipykernel_50/3480704817.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Smoke test PASS | scores:",sc.detach().float().cpu().tolist(),"loss:",float(loss))


## 6. Ranking training

Each query contributes positives and BM25 hard negatives. We use a combined pairwise softplus ranking loss and binary classification loss. This helps optimize ordering while keeping score calibration useful for Dev fusion.

In [35]:
trainable=[p for p in model.parameters() if p.requires_grad]+list(rank_head.parameters())
optimizer=torch.optim.AdamW([
    {"params":[p for p in model.parameters() if p.requires_grad],"lr":LR_LORA},
    {"params":rank_head.parameters(),"lr":LR_HEAD}
],weight_decay=WEIGHT_DECAY,betas=(0.9,0.999))
scaler=torch.amp.GradScaler("cuda")
micro_batches_per_epoch=math.ceil(len(train_groups)/BATCH_GROUPS)
steps_per_epoch=math.ceil(micro_batches_per_epoch/GRAD_ACCUM)
total_steps=steps_per_epoch*EPOCHS
warmup=max(1,int(total_steps*WARMUP_FRAC))
def lr_scale(step):
    if step<warmup: return max(0.05,(step+1)/warmup)
    p=(step-warmup)/max(1,total_steps-warmup)
    return 0.1+0.9*0.5*(1+math.cos(math.pi*min(1,p)))
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_scale)

def make_group_items(groups):
    items=[]
    for qid,qtext,cids,posset in groups:
        pos=[c for c in cids if c in posset]
        negs=[c for c in cids if c not in posset]       # ordered hardest -> easier (BM25 rank)
        n_hard=int(round(NEGATIVES_PER_QUERY*HARD_FRAC))
        hard=negs[:15]; rest=negs[15:]
        pick=random.sample(hard,min(n_hard,len(hard)))
        remaining=[c for c in negs if c not in pick]
        pick+=random.sample(remaining,min(NEGATIVES_PER_QUERY-len(pick),len(remaining)))
        for cid in pos+pick:
            items.append((qtext,txt(train_c[cid]),float(cid in posset),qid))
    return items

best_dev=-1.0
history=[]
best_path=CKPT_DIR/"model2_qlora_best.pt"
last_path=CKPT_DIR/"model2_qlora_last.pt"

def save_state(path,epoch,dev_f1):
    torch.save({"model":model.state_dict(),"rank_head":rank_head.state_dict(),
                "epoch":epoch,"dev_f1":dev_f1,"config":{"rank":RANK,"alpha":ALPHA,
                "pair_max_len":PAIR_MAX_LEN,"query_max_tokens":QUERY_MAX_TOKENS}},path)

for epoch in range(1,EPOCHS+1):
    model.train(); rank_head.train()
    order=list(range(len(train_groups))); random.shuffle(order)
    losses=[]; t0=time.time()
    optimizer.zero_grad(set_to_none=True)
    for step0 in range(0,len(order),BATCH_GROUPS):
        group_batch=[train_groups[j] for j in order[step0:step0+BATCH_GROUPS]]
        items=make_group_items(group_batch)
        ids,mask,starts,labels,gids=collate_pairs(items)
        with torch.autocast("cuda",dtype=torch.float16):
            scores=decoder_pair_scores(ids,mask,starts)
            bce=F.binary_cross_entropy_with_logits(scores,labels)
            pair_terms=[]; list_terms=[]
            for gid in set(gids):
                ix=[i for i,g in enumerate(gids) if g==gid]
                pos=[scores[i] for i in ix if labels[i]>0.5]
                neg=[scores[i] for i in ix if labels[i]<=0.5]
                if pos and neg:
                    p=torch.stack(pos); n=torch.stack(neg)
                    pair_terms.append(F.softplus(-(p[:,None]-n[None,:])).mean())
                    # multi-positive listwise softmax loss
                    allsc=torch.cat([p,n]).float()
                    list_terms.append(-(torch.logsumexp(p.float(),0)-torch.logsumexp(allsc,0)))
            pair_loss=torch.stack(pair_terms).mean() if pair_terms else scores.sum()*0
            list_loss=torch.stack(list_terms).mean() if list_terms else scores.sum()*0
            loss=(0.2*bce+0.4*pair_loss+0.4*list_loss)/GRAD_ACCUM
        scaler.scale(loss).backward()
        if ((step0//BATCH_GROUPS)+1)%GRAD_ACCUM==0 or step0+BATCH_GROUPS>=len(order):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable,1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        losses.append(float(loss.detach())*GRAD_ACCUM)
        if (step0//BATCH_GROUPS+1)%100==0:
            print(f"epoch {epoch}/{EPOCHS} | groups {min(step0+BATCH_GROUPS,len(order))}/{len(order)} | loss {np.mean(losses[-100:]):.4f} | {time.time()-t0:.0f}s",flush=True)
    print(f"Epoch {epoch} train loss={np.mean(losses):.5f} elapsed={(time.time()-t0)/60:.1f} min")
    save_state(last_path,epoch,-1)
    save_state(CKPT_DIR/f"epoch_{epoch}.pt",epoch,-1)
    gc.collect(); torch.cuda.empty_cache()
    # Dev evaluation below sets best_dev and saves best checkpoint.
    print("Epoch training complete; run Dev ranking evaluation in next cell.")

epoch 1/4 | groups 200/817 | loss 0.8510 | 77s
epoch 1/4 | groups 400/817 | loss 0.7472 | 140s
epoch 1/4 | groups 600/817 | loss 0.7785 | 201s
epoch 1/4 | groups 800/817 | loss 0.6884 | 262s
Epoch 1 train loss=0.76346 elapsed=4.4 min
Epoch training complete; run Dev ranking evaluation in next cell.
epoch 2/4 | groups 200/817 | loss 0.6301 | 58s
epoch 2/4 | groups 400/817 | loss 0.6695 | 117s
epoch 2/4 | groups 600/817 | loss 0.6012 | 175s
epoch 2/4 | groups 800/817 | loss 0.6440 | 233s
Epoch 2 train loss=0.63584 elapsed=4.0 min
Epoch training complete; run Dev ranking evaluation in next cell.
epoch 3/4 | groups 200/817 | loss 0.5604 | 58s
epoch 3/4 | groups 400/817 | loss 0.4872 | 116s
epoch 3/4 | groups 600/817 | loss 0.4740 | 174s
epoch 3/4 | groups 800/817 | loss 0.5144 | 232s
Epoch 3 train loss=0.50693 elapsed=4.0 min
Epoch training complete; run Dev ranking evaluation in next cell.
epoch 4/4 | groups 200/817 | loss 0.4264 | 57s
epoch 4/4 | groups 400/817 | loss 0.3622 | 115s
epoch

## 7. Dev retrieval evaluation and checkpoint selection

BM25 retrieves the top 100 Dev candidates. The neural reranker scores the shortlist, and the fusion coefficient is selected only on Dev Micro-F1@7.

In [38]:

# ============================================================
# PHASE 7 — DEV EVALUATION + BEST EPOCH SELECTION
# Model 2 | QLoRA | LegalBPE | BM25 + Neural Fusion
# ============================================================

import gc
import json
import torch
import numpy as np

# ------------------------------------------------------------
# 1. Build Dev BM25 index
# ------------------------------------------------------------
dev_cids = list(dev_c.keys())
bm25_dev = BM25Okapi([toks(txt(dev_c[c])) for c in dev_cids])

# ------------------------------------------------------------
# 2. Ensure model has the exact QLoRA architecture
# ------------------------------------------------------------
def ensure_qlora_architecture(reference_state):
    global model

    current_keys = set(model.state_dict().keys())
    saved_keys = set(reference_state.keys())

    if current_keys == saved_keys:
        print("Current model architecture matches saved QLoRA checkpoint.")
        return

    print("Current model does not match saved QLoRA checkpoint.")
    print("Reconstructing Model 2 from Phase-1 checkpoint...")

    # Release the incorrectly structured model.
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # Reload original Phase-1 Model 2 weights.
    base_ckpt = safe_torch_load(CHECKPOINT_PATH)

    fresh_model = DecoderOnlyLM(
        len(tokenizer),
        D_MODEL,
        N_LAYERS,
        N_HEADS,
        FFN_DIM,
        DROPOUT_BASE,
        MAX_POSITIONS
    )

    fresh_model.load_state_dict(base_ckpt["model"], strict=True)
    fresh_model.to(DEVICE)

    del base_ckpt
    gc.collect()

    # Recreate exactly the same NF4 + QLoRA structure used in training.
    for p in fresh_model.parameters():
        p.requires_grad = False

    replaced = replace_target_linears(fresh_model)
    assert len(replaced) > 0, "No QLoRA target layers were reconstructed."

    # Keep the tied token embedding in FP16, as in training.
    fresh_model.token_emb.weight.data = (
        fresh_model.token_emb.weight.data.to(torch.float16)
    )

    model = fresh_model
    gc.collect()
    torch.cuda.empty_cache()

    # Verify that the reconstructed architecture matches the checkpoint.
    rebuilt_keys = set(model.state_dict().keys())

    missing = saved_keys - rebuilt_keys
    unexpected = rebuilt_keys - saved_keys

    if missing or unexpected:
        raise RuntimeError(
            "QLoRA architecture mismatch after reconstruction.\n"
            f"Missing keys: {list(missing)[:15]}\n"
            f"Unexpected keys: {list(unexpected)[:15]}"
        )

    print("QLoRA architecture successfully reconstructed.")
    print("Quantized projection layers:", len(replaced))


# ------------------------------------------------------------
# 3. Load a complete epoch checkpoint safely
# ------------------------------------------------------------
def load_epoch_checkpoint(state):
    global model, rank_head

    saved = state["model"]

    # Keep only trained LoRA adapter parameters.
    # Do not reload frozen NF4 base weights or quantization metadata.
    adapter_state = {
        k: v for k, v in saved.items()
        if "lora_" in k.lower()
    }

    if not adapter_state:
        raise RuntimeError(
            "No LoRA parameters found in checkpoint. "
            "Print checkpoint keys before proceeding."
        )

    # Load adapters into the existing quantized model.
    result = model.load_state_dict(adapter_state, strict=False)

    # Verify that every saved adapter parameter was loaded.
    unexpected_adapter = [
        k for k in result.unexpected_keys
        if "lora_" in k.lower()
    ]

    if unexpected_adapter:
        raise RuntimeError(
            f"LoRA key mismatch: {unexpected_adapter[:10]}"
        )

    loaded_adapter_keys = set(adapter_state.keys())

    model_adapter_keys = {
        k for k in model.state_dict().keys()
        if "lora_" in k.lower()
    }

    missing_adapter = model_adapter_keys - loaded_adapter_keys

    if missing_adapter:
        raise RuntimeError(
            f"Some LoRA parameters were not loaded: "
            f"{list(missing_adapter)[:10]}"
        )

    rank_head.load_state_dict(state["rank_head"], strict=True)

    model.eval()
    rank_head.eval()

    print(
        f"Loaded {len(loaded_adapter_keys)} LoRA tensors "
        "and ranking head successfully."
    )


# ------------------------------------------------------------
# 4. Neural pair scoring
# ------------------------------------------------------------
@torch.no_grad()
def score_pairs(qtext, candidate_ids, candidate_dict,
                batch_size=EVAL_BATCH_SIZE):

    out = []
    model.eval()
    rank_head.eval()

    for s in range(0, len(candidate_ids), batch_size):
        chunk = candidate_ids[s:s + batch_size]

        items = [
            (qtext, txt(candidate_dict[cid]), 0.0, "eval")
            for cid in chunk
        ]

        ids, mask, starts, _, _ = collate_pairs(items)

        vals = decoder_pair_scores(ids, mask, starts)
        out.extend(vals.float().cpu().numpy().tolist())

    return out


# ------------------------------------------------------------
# 5. Micro-F1 / Precision / Recall
# ------------------------------------------------------------
def micro_metrics(gold, pred, k=7):
    tp = fp = fn = 0

    for qid, rel in gold.items():
        p = set(pred[qid][:k])
        r = set(map(str, rel))

        tp += len(p & r)
        fp += len(p - r)
        fn += len(r - p)

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)

    return {
        "micro_f1": f1,
        "micro_precision": precision,
        "micro_recall": recall,
        "tp": tp,
        "fp": fp,
        "fn": fn
    }


# ------------------------------------------------------------
# 6. Dev evaluation: BM25 + neural reranking + fusion
# ------------------------------------------------------------
def evaluate_dev():

    gold = {
        qid: [
            str(x)
            for x in rec.get("relevant_candidates", [])
            if str(x) in dev_c
        ]
        for qid, rec in dev_q.items()
    }

    bm25_preds = {}
    candidate_cache = {}

    for qid, rec in dev_q.items():
        ranked = bm25_rank(
            txt(rec),
            bm25_dev,
            dev_cids,
            TOP_K_BM25_EVAL
        )

        candidate_cache[qid] = ranked
        bm25_preds[qid] = [x[0] for x in ranked[:7]]

    bm25_m = micro_metrics(gold, bm25_preds)
    print("\nDev BM25:", bm25_m)

    # Score each query/candidate pair only once.
    neural_scores = {}

    for qi, (qid, rec) in enumerate(dev_q.items(), 1):

        ranked = candidate_cache[qid]
        ids = [x[0] for x in ranked]

        ns = score_pairs(txt(rec), ids, dev_c)

        neural_scores[qid] = {
            cid: float(v)
            for cid, v in zip(ids, ns)
        }

        if qi % 100 == 0:
            print(
                f"Dev scored {qi}/{len(dev_q)} queries",
                flush=True
            )

    best = None

    for alpha in [
        0.0, 0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9, 1.0
    ]:

        pred = {}

        for qid, ranked in candidate_cache.items():

            bm = {cid: score for cid, score in ranked}
            ns = neural_scores[qid]

            ordered_ids = list(bm.keys())

            bv = np.array(
                [bm[c] for c in ordered_ids],
                dtype=np.float32
            )

            nv = np.array(
                [ns[c] for c in ordered_ids],
                dtype=np.float32
            )

            bnorm = (
                (bv - bv.min()) /
                (bv.max() - bv.min() + 1e-8)
            )

            nnorm = (
                (nv - nv.min()) /
                (nv.max() - nv.min() + 1e-8)
            )

            fused = {
                cid: float(
                    (1 - alpha) * bnorm[i]
                    + alpha * nnorm[i]
                )
                for i, cid in enumerate(ordered_ids)
            }

            pred[qid] = sorted(
                fused,
                key=fused.get,
                reverse=True
            )[:7]

        metrics = micro_metrics(gold, pred)

        print(
            f"alpha={alpha:.1f} | "
            f"F1={metrics['micro_f1']:.5f} | "
            f"P={metrics['micro_precision']:.5f} | "
            f"R={metrics['micro_recall']:.5f}"
        )

        if best is None or metrics["micro_f1"] > best[0]:
            best = (
                metrics["micro_f1"],
                alpha,
                metrics,
                pred
            )

    return bm25_m, best


# ------------------------------------------------------------
# 7. Evaluate every saved epoch and select best checkpoint
# ------------------------------------------------------------
dev_results = []

best_dev = -1.0
best_alpha = 0.0

best_path = CKPT_DIR / "model2_qlora_best.pt"
best_predictions_path = PRED_DIR / "dev_best_predictions.json"

for ep in range(1, EPOCHS + 1):

    ep_path = CKPT_DIR / f"epoch_{ep}.pt"

    if not ep_path.exists():
        print("Missing epoch snapshot:", ep_path)
        continue

    print("\n" + "=" * 60)
    print(f"EVALUATING EPOCH {ep}/{EPOCHS}")
    print("=" * 60)

    state = torch.load(
        ep_path,
        map_location=DEVICE,
        weights_only=False
    )

    load_epoch_checkpoint(state)

    bm, best = evaluate_dev()

    dev_results.append({
        "epoch": ep,
        "f1": best[0],
        "alpha": best[1],
        "metrics": best[2]
    })

    print(f"\nEpoch {ep} best Dev result:")
    print("F1:", best[0])
    print("Alpha:", best[1])
    print("Metrics:", best[2])

    if best[0] > best_dev:

        best_dev = best[0]
        best_alpha = best[1]

        save_state(best_path, ep, best_dev)

        with open(best_predictions_path, "w") as f:
            json.dump(best[3], f, indent=2)

        print("\nNEW BEST CHECKPOINT SAVED")
        print("Epoch:", ep)
        print("Best Dev F1:", best_dev)
        print("Best alpha:", best_alpha)

    del state
    gc.collect()
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 8. Final selection summary
# ------------------------------------------------------------
if not dev_results:
    raise RuntimeError(
        "No epoch checkpoints were evaluated. "
        "Check CKPT_DIR and epoch_1.pt ... epoch_4.pt."
    )

print("\n" + "=" * 60)
print("FINAL BEST EPOCH")
print("=" * 60)

for result in dev_results:
    print(
        f"Epoch {result['epoch']} | "
        f"F1={result['f1']:.5f} | "
        f"alpha={result['alpha']:.1f}"
    )

best_result = max(dev_results, key=lambda x: x["f1"])

print("\nSelected epoch:", best_result["epoch"])
print("Best Dev Micro-F1@7:", best_result["f1"])
print("Selected fusion alpha:", best_result["alpha"])
print("Best checkpoint:", best_path)
print("Best Dev predictions:", best_predictions_path)



EVALUATING EPOCH 1/4
Loaded 104 LoRA tensors and ranking head successfully.

Dev BM25: {'micro_f1': 0.19455252918287938, 'micro_precision': 0.211864406779661, 'micro_recall': 0.17985611510791366, 'tp': 175, 'fp': 651, 'fn': 798}
Dev scored 100/118 queries
alpha=0.0 | F1=0.19455 | P=0.21186 | R=0.17986
alpha=0.1 | F1=0.20456 | P=0.22276 | R=0.18911
alpha=0.2 | F1=0.21123 | P=0.23002 | R=0.19527
alpha=0.3 | F1=0.19011 | P=0.20702 | R=0.17575
alpha=0.4 | F1=0.16454 | P=0.17918 | R=0.15211
alpha=0.5 | F1=0.13563 | P=0.14770 | R=0.12539
alpha=0.6 | F1=0.12674 | P=0.13801 | R=0.11716
alpha=0.7 | F1=0.11451 | P=0.12470 | R=0.10586
alpha=0.8 | F1=0.10895 | P=0.11864 | R=0.10072
alpha=0.9 | F1=0.10673 | P=0.11622 | R=0.09866
alpha=1.0 | F1=0.10339 | P=0.11259 | R=0.09558

Epoch 1 best Dev result:
F1: 0.21122846025569758
Alpha: 0.2
Metrics: {'micro_f1': 0.21122846025569758, 'micro_precision': 0.23002421307506055, 'micro_recall': 0.19527235354573483, 'tp': 190, 'fp': 636, 'fn': 783}

NEW BEST CH

## 8. Professor-provided official evaluator — Dev scoring

This cell writes the exact `evaluate_phase7.py` source supplied earlier, then runs it on the best Dev predictions. It does not substitute a locally reimplemented metric.

In [39]:
# The evaluator source below is copied verbatim from the professor-provided script.
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for CS-781 Phase 7 — Prior Case Retrieval (PCR).\n\nIt uses the same validation rules and Micro-F1@7 / Micro-Precision@7 /\nMicro-Recall@7 definitions as the Codabench scorer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nK = 7\n\n\ndef reject_constant(value: str) -> None:\n    raise ValueError(f"Non-standard JSON numeric value \'{value}\' is not allowed.")\n\n\ndef pairs_no_duplicates(pairs):\n    result = {}\n    duplicates = []\n    for key, value in pairs:\n        if key in result:\n            duplicates.append(key)\n        result[key] = value\n    if duplicates:\n        raise ValueError(f"Duplicate JSON object key(s) detected: {list(dict.fromkeys(duplicates))}")\n    return result\n\n\ndef load_json(path: Path) -> Any:\n    try:\n        with path.open("r", encoding="utf-8") as f:\n            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n    except json.JSONDecodeError as exc:\n        raise ValueError(\n            f"Invalid JSON in {path} at line {exc.lineno}, column {exc.colno}: {exc.msg}"\n        ) from exc\n\n\ndef load_truth(path: Path) -> dict[str, list[str]]:\n    data = load_json(path)\n    if not isinstance(data, dict) or not data:\n        raise ValueError("Truth must be a non-empty JSON object.")\n    out: dict[str, list[str]] = {}\n    for query_id, value in data.items():\n        if isinstance(value, dict) and "relevant_candidates" in value:\n            labels = value["relevant_candidates"]\n        else:\n            labels = value\n        if not isinstance(query_id, str) or not query_id.strip():\n            raise ValueError("Truth contains an empty/invalid query ID.")\n        if not isinstance(labels, list):\n            raise ValueError(f"Truth for query \'{query_id}\' must contain a list of relevant candidate IDs.")\n        cleaned=[]; seen=set()\n        for idx,candidate_id in enumerate(labels):\n            if not isinstance(candidate_id,str) or not candidate_id.strip():\n                continue\n            if candidate_id in seen:\n                raise ValueError(f"Truth for query \'{query_id}\' contains duplicate candidate ID \'{candidate_id}\'.")\n            seen.add(candidate_id); cleaned.append(candidate_id)\n        out[query_id]=cleaned\n    return out\n\n\ndef load_candidate_ids(path: Path) -> set[str]:\n    data = load_json(path)\n    if not isinstance(data, list) or not data:\n        raise ValueError("candidate_ids must be a non-empty JSON list.")\n    ids=[]; seen=set()\n    for idx,v in enumerate(data):\n        if not isinstance(v,str) or not v.strip(): raise ValueError(f"candidate_ids index {idx} is invalid.")\n        if v in seen: raise ValueError(f"candidate_ids contains duplicate ID \'{v}\'.")\n        seen.add(v); ids.append(v)\n    return set(ids)\n\n\ndef load_prediction(path: Path) -> Any:\n    return load_json(path)\n\n\ndef validate(pred: Any, gold: dict[str,list[str]], candidates: set[str]) -> list[str]:\n    errors=[]\n    if not isinstance(pred,dict): return ["prediction.json must be a JSON object mapping query IDs to ranked lists."]\n    gold_ids=set(gold); pred_ids=set(pred)\n    for q in sorted(gold_ids-pred_ids): errors.append(f"Missing required query ID \'{q}\'.")\n    for q in sorted(pred_ids-gold_ids): errors.append(f"Unknown query ID \'{q}\'.")\n    for q in sorted(gold_ids & pred_ids):\n        vals=pred[q]\n        if not isinstance(vals,list):\n            errors.append(f"Query \'{q}\': prediction must be a list."); continue\n        if len(vals)!=K: errors.append(f"Query \'{q}\': expected exactly {K} candidates, got {len(vals)}.")\n        seen=set()\n        for i,c in enumerate(vals):\n            if not isinstance(c,str) or not c.strip(): errors.append(f"Query \'{q}\': index {i} is not a non-empty string."); continue\n            if c in seen: errors.append(f"Query \'{q}\': duplicate candidate ID \'{c}\'.")\n            seen.add(c)\n            if c not in candidates: errors.append(f"Query \'{q}\': candidate ID \'{c}\' is outside the candidate pool.")\n    return errors\n\n\ndef score(gold:dict[str,list[str]], pred:dict[str,list[str]]) -> dict[str,float|int]:\n    tp=fp=fn=0\n    for q,labels in gold.items():\n        g=set(labels); p=set(pred[q][:K]); tp+=len(g&p); fp+=len(p-g); fn+=len(g-p)\n    precision=tp/(tp+fp) if tp+fp else 0.0\n    recall=tp/(tp+fn) if tp+fn else 0.0\n    f1=(2*tp)/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0\n    return {"primary_metric_1":f1,"secondary_metric_1":precision,"secondary_metric_2":recall,"tp":tp,"fp":fp,"fn":fn}\n\n\ndef parse_args():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--truth",type=Path,required=True,help="gold.json or Phase-7 dev.json with relevant_candidates")\n    p.add_argument("--prediction",type=Path,required=True)\n    p.add_argument("--candidate-ids",type=Path,required=True,help="candidate_ids.json")\n    p.add_argument("--output",type=Path,default=None,help="Optional output scores JSON path")\n    return p.parse_args()\n\n\ndef main():\n    a=parse_args()\n    gold=load_truth(a.truth); candidates=load_candidate_ids(a.candidate_ids); pred=load_prediction(a.prediction)\n    errs=validate(pred,gold,candidates)\n    if errs:\n        print("LOCAL VALIDATION FAILED")\n        for i,e in enumerate(errs,1): print(f"  {i}. {e}")\n        print(f"Total validation errors: {len(errs)}")\n        raise SystemExit(1)\n    result=score(gold,pred)\n    scores={k:result[k] for k in ("primary_metric_1","secondary_metric_1","secondary_metric_2")}\n    print("Validation passed.")\n    print(f"Micro-F1@7: {scores[\'primary_metric_1\']:.10f}")\n    print(f"Micro-Precision@7: {scores[\'secondary_metric_1\']:.10f}")\n    print(f"Micro-Recall@7: {scores[\'secondary_metric_2\']:.10f}")\n    print(f"TP={result[\'tp\']} FP={result[\'fp\']} FN={result[\'fn\']}")\n    if a.output:\n        a.output.parent.mkdir(parents=True,exist_ok=True)\n        with a.output.open(\'w\',encoding=\'utf-8\') as f: json.dump(scores,f,indent=2); f.write(\'\\n\')\n        print(f"Wrote {a.output}")\n\nif __name__==\'__main__\': main()\n'
EVALUATOR_PATH = ROOT / "evaluate_phase7.py"
EVALUATOR_PATH.write_text(EVALUATOR_SOURCE, encoding="utf-8")

DEV_TRUTH_PATH = ROOT / "dev_truth.json"
DEV_CANDIDATE_IDS_PATH = ROOT / "dev_candidate_ids.json"
DEV_PREDICTION_PATH = PRED_DIR / "dev_best_predictions.json"
DEV_EVAL_OUTPUT = ROOT / "official_dev_scores.json"

DEV_TRUTH_PATH.write_text(json.dumps(dev_q, ensure_ascii=False), encoding="utf-8")
DEV_CANDIDATE_IDS_PATH.write_text(
    json.dumps(list(dev_c.keys()), ensure_ascii=False, indent=2), encoding="utf-8"
)
if not DEV_PREDICTION_PATH.is_file():
    raise FileNotFoundError(
        f"Best Dev predictions not found: {DEV_PREDICTION_PATH}. Run the Dev selection cell first."
    )

cmd = [
    sys.executable, str(EVALUATOR_PATH),
    "--truth", str(DEV_TRUTH_PATH),
    "--prediction", str(DEV_PREDICTION_PATH),
    "--candidate-ids", str(DEV_CANDIDATE_IDS_PATH),
    "--output", str(DEV_EVAL_OUTPUT),
]
print("\\nRunning professor's official evaluator on Dev:", stamp())
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Professor's official evaluator FAILED on Dev predictions.")
print("Official Dev evaluation passed. Scores saved:", DEV_EVAL_OUTPUT)


\nRunning professor's official evaluator on Dev: 2026-11-06 13:59:54
Validation passed.
Micro-F1@7: 0.2190105614
Micro-Precision@7: 0.2384987893
Micro-Recall@7: 0.2024665982
TP=197 FP=629 FN=776
Wrote /kaggle/working/phase7_model2_qlora/official_dev_scores.json

Official Dev evaluation passed. Scores saved: /kaggle/working/phase7_model2_qlora/official_dev_scores.json


## 9. Test inference and submission ZIP\n\nRestore the best Dev-selected checkpoint and fusion coefficient, rank each Test query only against the Test candidate pool, then write `prediction.json` and a ZIP with that file at its root.

In [41]:

# ============================================================
# PHASE 7 — FINAL TEST INFERENCE + SUBMISSION ZIP
# Model 2 | QLoRA | BM25 + Neural Fusion
# ============================================================

import json
import zipfile
import numpy as np
import torch

# ------------------------------------------------------------
# 1. Load the best Dev-selected checkpoint
# ------------------------------------------------------------
assert best_path.exists(), f"Best checkpoint not found: {best_path}"

state = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False
)

# Uses the adapter-only loader defined in the Dev evaluation cell.
load_epoch_checkpoint(state)

del state

model.eval()
rank_head.eval()

print("Best checkpoint loaded successfully.")
print("Selected fusion alpha:", best_alpha)

# ------------------------------------------------------------
# 2. Build Test BM25 index
# ------------------------------------------------------------
test_cids = list(test_c.keys())

assert len(test_cids) >= 7, (
    "Test candidate pool contains fewer than 7 candidates."
)

print("Building Test BM25 index...")

bm25_test = BM25Okapi([
    toks(txt(test_c[c]))
    for c in test_cids
])

print("Test BM25 index ready.")
print("Test queries:", len(test_q))
print("Test candidates:", len(test_cids))

# ------------------------------------------------------------
# 3. Generate Test predictions
# ------------------------------------------------------------
test_predictions = {}

for qi, (qid, rec) in enumerate(test_q.items(), 1):

    # BM25 candidate retrieval
    ranked = bm25_rank(
        txt(rec),
        bm25_test,
        test_cids,
        TOP_K_BM25_EVAL
    )

    ids = [x[0] for x in ranked]

    # Neural reranking
    ns = score_pairs(
        txt(rec),
        ids,
        test_c
    )

    # BM25 scores
    bm = {cid: score for cid, score in ranked}

    bv = np.array(
        [bm[c] for c in ids],
        dtype=np.float32
    )

    nv = np.array(
        ns,
        dtype=np.float32
    )

    # Per-query min-max normalization
    bnorm = (
        (bv - bv.min()) /
        (bv.max() - bv.min() + 1e-8)
    )

    nnorm = (
        (nv - nv.min()) /
        (nv.max() - nv.min() + 1e-8)
    )

    # BM25 + neural score fusion
    fused = {
        cid: float(
            (1 - best_alpha) * bnorm[i]
            + best_alpha * nnorm[i]
        )
        for i, cid in enumerate(ids)
    }

    chosen = sorted(
        fused,
        key=fused.get,
        reverse=True
    )[:7]

    # Fill if retrieval returned fewer than 7 candidates
    if len(chosen) < 7:
        chosen_set = set(chosen)

        chosen += [
            c for c in test_cids
            if c not in chosen_set
        ][:7 - len(chosen)]

    test_predictions[qid] = chosen

    if qi % 10 == 0 or qi == len(test_q):
        print(
            f"Test ranked {qi}/{len(test_q)} queries",
            flush=True
        )

# ------------------------------------------------------------
# 4. Validate predictions
# ------------------------------------------------------------
assert set(test_predictions) == set(test_q), (
    "Prediction query IDs do not match Test query IDs."
)

for qid, ids in test_predictions.items():

    assert len(ids) == 7, (
        f"{qid}: Expected 7 predictions, got {len(ids)}"
    )

    assert len(set(ids)) == 7, (
        f"{qid}: Duplicate candidate IDs found."
    )

    assert set(ids).issubset(set(test_c)), (
        f"{qid}: Prediction contains an invalid candidate ID."
    )

print("All Test predictions passed validation.")

# ------------------------------------------------------------
# 5. Save prediction.json
# ------------------------------------------------------------
pred_path = ROOT / "prediction.json"

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# 6. Create submission ZIP
# ------------------------------------------------------------
zip_path = ROOT / "submission.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        pred_path,
        arcname="prediction.json"
    )

# ------------------------------------------------------------
# 7. Final ZIP validation
# ------------------------------------------------------------
with zipfile.ZipFile(zip_path, "r") as z:

    assert z.namelist() == ["prediction.json"]

    # Verify the archived JSON is readable.
    archived_predictions = json.loads(
        z.read("prediction.json").decode("utf-8")
    )

    assert archived_predictions == test_predictions

print("\n" + "=" * 55)
print("SUBMISSION CREATED AND VALIDATED")
print("=" * 55)
print("Submission:", zip_path)
print("Prediction file:", pred_path)
print("Queries:", len(test_predictions))
print("Predictions per query: 7")
print("Fusion alpha:", best_alpha)
print("ZIP structure: prediction.json")


Loaded 104 LoRA tensors and ranking head successfully.
Best checkpoint loaded successfully.
Selected fusion alpha: 0.2
Building Test BM25 index...
Test BM25 index ready.
Test queries: 237
Test candidates: 1727
Test ranked 10/237 queries
Test ranked 20/237 queries
Test ranked 30/237 queries
Test ranked 40/237 queries
Test ranked 50/237 queries
Test ranked 60/237 queries
Test ranked 70/237 queries
Test ranked 80/237 queries
Test ranked 90/237 queries
Test ranked 100/237 queries
Test ranked 110/237 queries
Test ranked 120/237 queries
Test ranked 130/237 queries
Test ranked 140/237 queries
Test ranked 150/237 queries
Test ranked 160/237 queries
Test ranked 170/237 queries
Test ranked 180/237 queries
Test ranked 190/237 queries
Test ranked 200/237 queries
Test ranked 210/237 queries
Test ranked 220/237 queries
Test ranked 230/237 queries
Test ranked 237/237 queries
All Test predictions passed validation.

SUBMISSION CREATED AND VALIDATED
Submission: /kaggle/working/phase7_model2_qlora/submi

## 10. Official Test structural validation + final ZIP

The professor's evaluator cannot calculate Test F1 because Test labels are private. This cell imports the exact evaluator written above and reuses its `validate()` function against an empty-label truth map to check query coverage, exactly 7 unique IDs, valid strings, and candidate-pool membership.

In [42]:
# Structural validation only — no Test relevance labels or Test F1 are available.
TEST_CANDIDATE_IDS_PATH = ROOT / "test_candidate_ids.json"
TEST_CANDIDATE_IDS_PATH.write_text(
    json.dumps(list(test_c.keys()), ensure_ascii=False, indent=2), encoding="utf-8"
)

spec = __import__("importlib.util").util.spec_from_file_location(
    "evaluate_phase7", EVALUATOR_PATH
)
evaluator = __import__("importlib.util").util.module_from_spec(spec)
spec.loader.exec_module(evaluator)

test_truth_for_validation = {str(qid): [] for qid in test_q}
test_candidate_set = evaluator.load_candidate_ids(TEST_CANDIDATE_IDS_PATH)
test_prediction_obj = evaluator.load_prediction(pred_path)
test_errors = evaluator.validate(
    test_prediction_obj, test_truth_for_validation, test_candidate_set
)
if test_errors:
    print("TEST SUBMISSION VALIDATION FAILED")
    for i, err in enumerate(test_errors, 1):
        print(f"  {i}. {err}")
    raise RuntimeError("Test submission failed professor's structural validation.")

# Verify the ZIP contains exactly one root-level prediction.json and matches the file.
with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()
    assert names == ["prediction.json"], f"Unexpected ZIP contents: {names}"
    zipped_prediction = json.loads(z.read("prediction.json").decode("utf-8"))
    assert zipped_prediction == test_prediction_obj

print("TEST SUBMISSION STRUCTURAL VALIDATION PASSED")
print("Checked: exact Test query IDs | exactly 7 IDs/query | no duplicates | all IDs in Test candidate pool.")
print("No Test score is calculated because Test relevance labels are private.")
print("Final ZIP to upload:", zip_path)
print("ZIP size: %.2f MB" % (zip_path.stat().st_size / (1024**2)))


TEST SUBMISSION STRUCTURAL VALIDATION PASSED
Checked: exact Test query IDs | exactly 7 IDs/query | no duplicates | all IDs in Test candidate pool.
No Test score is calculated because Test relevance labels are private.
Final ZIP to upload: /kaggle/working/phase7_model2_qlora/submission.zip
ZIP size: 0.01 MB


In [43]:
import shutil
from pathlib import Path

ROOT = Path("/kaggle/working/phase7_model2_qlora")

# Flatten all files from subfolders into ROOT.
for folder in list(ROOT.iterdir()):
    if folder.is_dir():
        for file in folder.rglob("*"):
            if file.is_file():
                # Preserve folder name to avoid filename collisions.
                prefix = "_".join(file.relative_to(ROOT).parts[:-1])
                destination = ROOT / (
                    f"{prefix}_{file.name}" if prefix else file.name
                )

                if destination.exists():
                    destination = ROOT / (
                        f"{prefix}_{file.stem}_copy{file.suffix}"
                    )

                shutil.move(str(file), str(destination))

        shutil.rmtree(folder)

print("Cleaned output directory:")
for item in sorted(ROOT.iterdir()):
    print(item.name)

Cleaned output directory:
__pycache___evaluate_phase7.cpython-313.pyc
cache_train_candidate_legalbpe_tokens.pkl
checkpoints_epoch_1.pt
checkpoints_epoch_2.pt
checkpoints_epoch_3.pt
checkpoints_epoch_4.pt
checkpoints_model2_qlora_best.pt
checkpoints_model2_qlora_last.pt
dev_candidate_ids.json
dev_truth.json
evaluate_phase7.py
official_dev_scores.json
prediction.json
predictions_dev_best_predictions.json
submission.zip
test_candidate_ids.json
train_candidate_tokens.pkl


## Run checklist and important notes

1. Select **GPU T4** (or better) in Kaggle.
2. Attach the exact Phase-1 Model 2 checkpoint and matching `tokenizer.json`. If automatic discovery finds multiple files, set `CHECKPOINT_PATH` and `TOKENIZER_PATH` in Cell 1.
3. Run top-to-bottom. Confirm the strict checkpoint load succeeds and the NF4 forward/backward smoke test passes before training.
4. Training saves one snapshot per epoch. Dev selection evaluates all four snapshots and chooses the best checkpoint + fusion weight using Dev only.
5. Run the official evaluator cell after Dev selection; it writes the exact professor-provided `evaluate_phase7.py`, scores Dev predictions, and saves `official_dev_scores.json`.
6. Test inference uses only the Test candidate pool. The final validation cell applies the evaluator's own structural validation to Test predictions (no hidden-label score is possible) and verifies the ZIP.

### Improvements incorporated from your updated notebook
- Four epochs; LoRA rank 32 / alpha 64; higher LoRA LR.
- Larger sparse-BM25 hard-negative pool (60), with 7 negatives re-sampled every epoch; BM25 scoring uses the earlier optimized sparse-matrix approach and exact LegalBPE IDs.
- 512-token pair context with head+tail truncation and tokenization caching.
- Combined BCE + pairwise + multi-positive listwise losses.
- More complete Dev fusion grid and per-epoch checkpoint selection.
- Scheduler corrected to count optimizer steps after gradient accumulation.

### Memory/runtime fallback
If the QLoRA smoke test or first training batch runs out of memory:
- Set `BATCH_GROUPS = 1` and `GRAD_ACCUM = 8`.
- Reduce `EVAL_BATCH_SIZE` to 8.
- If needed, reduce `PAIR_MAX_LEN` to 448 and recompute `CAND_MAX_TOKENS` from the existing formula.

Do not change the tokenizer, load any non-Phase-1 pretrained weights, or use Dev/Test candidates as training negatives. The previous best experiment score is **0.3473**; this experiment is intended to beat it, but no improvement is guaranteed. The notebook has been syntax-checked locally, **not executed on Kaggle**, so runtime and QLoRA compatibility still need the Kaggle smoke test.


### Tokenizer consistency (important)
All retrieval-side tokenization now calls `tokenizer.encode(...)` from the loaded Phase-1 Model 2 `LegalBPE` tokenizer. Train, Dev, and Test BM25 documents and queries therefore use the same LegalBPE vocabulary as the neural cross-encoder. No regex word tokenizer is used for BM25. The cached Train candidate tokens are fingerprinted against the tokenizer merges and candidate ID order.
